# Lend a GPU to karaokie

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/karaokie-app/colab/main/kaggle_worker.ipynb)

Preparing a song means separating the voice from the music and placing every
word of the lyrics against it — a few minutes of work for a graphics card.
This notebook borrows the one Kaggle gives you, puts it on the queue at
[karaokie.app](https://karaokie.app), and hands the finished songs back.

Nothing is installed on your own machine and there is no account to make with
karaokie. Kaggle needs two switches turned on first, and neither is on by
default.

---

### 1. Sign in to Kaggle

Signed out there is no accelerator and no internet switch at all — the panel on
the right offers only this:

<img src="https://karaokie.app/guide/kaggle-session.png" alt="Kaggle's Session options panel when signed out: Language and Environment, and a line inviting you to sign in for a GPU and an internet connection" width="300">

### 2. Make it yours

If this looks read-only, press **Copy & Edit** at the top right. You cannot run
somebody else's notebook, only your own copy of it.

### 3. Ask for a graphics card

Signed in, two more settings appear under that same **Session options**
heading. The first is **Accelerator ▸ GPU T4 x2**.
(P100 is just as good. Only one card is used either way — a song is one job.)

Without a card the work still runs, on the processor, and a song takes
something like ten times as long.

### 4. Turn the internet on

Same panel: **Internet ▸ On**. It is off by default, and nothing here works
without it — no download, and nothing sent back.

The switch needs a phone-verified Kaggle account. If it is greyed out, that is
why: your profile ▸ **Settings ▸ Phone verification**.

### 5. Press **Run All**

In the bar at the top. That runs both cells below, in order, which is the
entire job.

<img src="https://karaokie.app/guide/kaggle-toolbar.png" alt="Kaggle's notebook toolbar, with Run All in the middle" width="320">

### 6. Wait, and read the output

The first cell checks this machine and prints three lines: the card, the
internet, and whether YouTube will answer it. If any of them is wrong it says
what to do about it.

The second downloads the worker and starts it. The first few minutes are it
fetching its own Python, ffmpeg and the audio models — several GB, once — and
after that it prints a line for every song it prepares.

### 7. Leave this tab open

That is the whole job.

---

**To stop:** press the ■ beside the running cell, or **Run ▸ Stop session**.
Nothing needs tidying up — a song caught half-done goes back on the queue for
somebody else a few minutes later.


In [ ]:
# 1 — is this session any use?
#
# Three things have to be true, and two of them are switches in the panel on
# the right that are easy to forget. Better to find out here than three
# minutes into a song.
import subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

gpu = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
cards = gpu.stdout.strip().splitlines() if gpu.returncode == 0 else []
if cards:
    print('GPU       ', cards[0], f'({len(cards)} attached)' if len(cards) > 1 else '')
else:
    print('GPU        none. Settings > Accelerator > GPU T4 x2, then run this again.')
    print('           It will still work on the processor, only far more slowly.')

net = sh('curl -fsS -o /dev/null -w %{http_code} https://karaokie.app/install.sh')
if net.stdout.strip() != '200':
    print('Internet   off. Settings > Internet > On -- nothing can be downloaded')
    print('           without it. The switch needs a phone-verified account.')
else:
    print('Internet   on')
    print('YouTube    asking...')
    sh('pip -q install -U yt-dlp')
    # The same request the pipeline makes: music.youtube.com, over IPv4.
    probe = sh("yt-dlp -4 --skip-download --no-warnings --print '%(title)s' "
               'https://music.youtube.com/watch?v=sQtnhwU2R9Y')
    if probe.returncode == 0 and probe.stdout.strip():
        print('YouTube    ok --', probe.stdout.strip().splitlines()[-1])
    else:
        said = (probe.stderr or '').strip().splitlines()
        print('YouTube    would not answer this machine:')
        print('          ', said[-1] if said else 'no answer at all')
        print()
        print('           Kaggle addresses are datacentre addresses, and those are')
        print('           the ones asked to prove they are not a robot. A worker that')
        print('           cannot fetch a song is no use to the queue: stop the session')
        print('           and start it again to be given a different machine.')


In [ ]:
# 2 — run the worker.
#
# One binary, checked against the published checksum and started. The first
# few minutes are it fetching its own Python, ffmpeg and the audio models;
# after that it takes a song, prepares it, uploads it and asks for another.
#
# Press the stop button on this cell to stop it.
import os, subprocess

card = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                      shell=True, capture_output=True, text=True).stdout.strip()
# How this machine appears in the pool at karaokie.app/worker.
os.environ['KARAOKIE_NAME'] = f'kaggle {card.splitlines()[0]}' if card else 'kaggle'
# Not /kaggle/working: everything under that is saved as this notebook's
# output and counts against its 20 GB, and none of this is worth keeping.
os.environ['KARAOKIE_DIR'] = '/root/.karaokie-agent'

!curl -fsSL https://karaokie.app/install.sh | sh


### What Kaggle gives you

About **30 GPU-hours a week**, which is the pleasant thing about it: the quota
is published rather than guessed at, and you can see what is left of it under
the accelerator menu. A session runs for up to nine hours with a GPU, and ends
roughly forty minutes after you close the tab.

Kaggle is for data-science work, and a notebook that does nothing but run a
background job for somebody else's website is not really what the quota is
for. Run it while you are around, stop it when you are not, and it stays what
it is meant to be — a machine you are using, doing something useful with the
part of it you are not.

### While it runs

Each song is logged as it goes. A long run fills this cell with a lot of text;
clearing the output does not interrupt the worker.

Who else is working, and every other way to run one:
[karaokie.app/worker](https://karaokie.app/worker).
